# Can we predict 12-month loan default?

**Question:** using only information available at application time, can we predict `defaulted` (1 = defaulted within 12 months of funding)?

**Plan:** validate data -> EDA -> hold out a test set -> compare a baseline, a logistic regression and gradient boosting with cross-validation on the training set -> evaluate once on the test set -> interpret and judge practical usefulness.

Defaults are rare (~10%), so accuracy is misleading. We focus on ROC AUC and PR AUC (average precision), plus lift when flagging the riskiest applicants.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve, brier_score_loss
from sklearn.inspection import permutation_importance
from sklearn.calibration import calibration_curve
SEED = 42
pd.set_option('display.width', 140)
df = pd.read_csv('data.csv'); dd = pd.read_csv('columns.csv')
dd

## 1. Data checks

In [ ]:
print(df.shape)
print('missing values:\n', df.isna().sum().to_string())
print('duplicate rows:', df.duplicated().sum(), '| duplicate ids:', df.application_id.duplicated().sum())
df.describe().T

In [ ]:
# Plausibility checks
print('employment_years > age-16:', (df.employment_years > df.age - 16).sum())
print('income <= 0:', (df.income_k <= 0).sum(), '| loan <= 0:', (df.loan_amount_k <= 0).sum(), '| debt_ratio outside [0,1]:', (~df.debt_ratio.between(0, 1)).sum())
print('employment_years == 0:', (df.employment_years == 0).sum())
print(df.region.value_counts().to_string())

## 2. Target and exploratory analysis

In [ ]:
rate = df.defaulted.mean()
print(f'Default rate: {rate:.2%} ({df.defaulted.sum()} of {len(df)})')
num = ['income_k', 'debt_ratio', 'credit_lines', 'age', 'employment_years', 'loan_amount_k']
fig, ax = plt.subplots(2, 3, figsize=(14, 7))
for a, c in zip(ax.ravel(), num):
    q = pd.qcut(df[c], 10, duplicates='drop')
    g = df.groupby(q, observed=True).defaulted.mean()
    a.bar(range(len(g)), g.values, color='#3b6ea5'); a.axhline(rate, color='k', ls='--', lw=1)
    a.set_title(f'Default rate by {c} decile'); a.set_xlabel('decile (low to high)'); a.set_ylabel('default rate')
plt.tight_layout(); plt.show()
print(df.groupby('region').defaulted.agg(['mean', 'count']).round(3))

In [ ]:
df[num + ['defaulted']].corr(method='spearman')['defaulted'].drop('defaulted').round(3)

`application_id` is an identifier and is excluded from modelling. Read the decile plots above for which predictors show a trend and which look flat.

## 3. Train/test split
Stratified 75/25 split. The test set is used once, at the end.

In [ ]:
feats = num + ['region']
X = df[feats]; y = df.defaulted
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEED)
print(Xtr.shape, Xte.shape, ytr.mean().round(4), yte.mean().round(4))

## 4. Model comparison (5-fold CV on the training set)

In [ ]:
pre_lr = ColumnTransformer([('n', StandardScaler(), num), ('c', OneHotEncoder(drop='first'), ['region'])])
pre_tree = ColumnTransformer([('n', 'passthrough', num), ('c', OneHotEncoder(), ['region'])])
models = {
    'baseline (prior)': Pipeline([('p', pre_lr), ('m', DummyClassifier(strategy='prior'))]),
    'logistic regression': Pipeline([('p', pre_lr), ('m', LogisticRegression(max_iter=1000))]),
    'gradient boosting': Pipeline([('p', pre_tree), ('m', HistGradientBoostingClassifier(learning_rate=0.05, max_iter=200, max_depth=3, min_samples_leaf=50, l2_regularization=1.0, random_state=SEED))]),
}
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
rows = {}
for n, m in models.items():
    r = cross_validate(m, Xtr, ytr, cv=cv, scoring={'auc': 'roc_auc', 'ap': 'average_precision', 'brier': 'neg_brier_score'})
    rows[n] = {'ROC AUC': r['test_auc'].mean(), 'AUC sd': r['test_auc'].std(), 'PR AUC': r['test_ap'].mean(), 'Brier': -r['test_brier'].mean()}
pd.DataFrame(rows).T.round(4)

If boosting does not clearly beat logistic regression, the signal is mostly additive and the simpler, more explainable model is preferable. The final model is chosen from these CV numbers, not from the test set.

## 5. Final evaluation on the held-out test set

In [ ]:
lr = models['logistic regression'].fit(Xtr, ytr)
gb = models['gradient boosting'].fit(Xtr, ytr)
res = {}
for n, m in [('logistic regression', lr), ('gradient boosting', gb)]:
    pp = m.predict_proba(Xte)[:, 1]
    res[n] = {'ROC AUC': roc_auc_score(yte, pp), 'PR AUC': average_precision_score(yte, pp), 'Brier': brier_score_loss(yte, pp)}
res['baseline (prior)'] = {'ROC AUC': 0.5, 'PR AUC': yte.mean(), 'Brier': brier_score_loss(yte, np.full(len(yte), ytr.mean()))}
print(pd.DataFrame(res).T.round(4))

# bootstrap CI for the logistic regression test AUC
p = lr.predict_proba(Xte)[:, 1]; yv = yte.values
rng = np.random.default_rng(SEED)
bs = []
for _ in range(500):
    i = rng.integers(0, len(yv), len(yv)); bs.append(roc_auc_score(yv[i], p[i]))
print('LR test AUC 95%% CI: %.3f - %.3f' % tuple(np.percentile(bs, [2.5, 97.5])))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
for n, m in [('logistic regression', lr), ('gradient boosting', gb)]:
    pp = m.predict_proba(Xte)[:, 1]
    f, t, _ = roc_curve(yte, pp); ax[0].plot(f, t, label=f'{n} (AUC {roc_auc_score(yte, pp):.3f})')
    pr, rc, _ = precision_recall_curve(yte, pp); ax[1].plot(rc, pr, label=n)
ax[0].plot([0, 1], [0, 1], 'k--', lw=1); ax[0].set(title='ROC', xlabel='FPR', ylabel='TPR'); ax[0].legend()
ax[1].axhline(yte.mean(), color='k', ls='--', lw=1); ax[1].set(title='Precision-recall', xlabel='recall', ylabel='precision'); ax[1].legend()
fp, mp = calibration_curve(yte, p, n_bins=8, strategy='quantile')
ax[2].plot(mp, fp, 'o-'); ax[2].plot([0, .4], [0, .4], 'k--', lw=1); ax[2].set(title='Calibration (logistic regression)', xlabel='predicted', ylabel='observed')
plt.tight_layout(); plt.show()

### Practical usefulness: ranking applicants by risk

In [ ]:
t = pd.DataFrame({'p': p, 'y': yv}).sort_values('p', ascending=False).reset_index(drop=True)
rows = []
for k in [0.05, 0.10, 0.20, 0.30, 0.50]:
    top = t.head(int(len(t) * k))
    rows.append({'top % flagged': int(k * 100), 'default rate in group': top.y.mean(), 'lift vs base': top.y.mean() / yv.mean(), 'share of all defaults caught': top.y.sum() / yv.sum()})
pd.DataFrame(rows).round(3)

## 6. What drives the prediction?

In [ ]:
coef = pd.Series(lr.named_steps['m'].coef_[0], index=lr.named_steps['p'].get_feature_names_out()).sort_values()
print('Standardised logistic coefficients (log-odds per 1 SD):'); print(coef.round(3).to_string())
pi = permutation_importance(lr, Xte, yte, scoring='roc_auc', n_repeats=20, random_state=SEED)
imp = pd.Series(pi.importances_mean, index=feats); sd = pd.Series(pi.importances_std, index=feats)
order = imp.sort_values().index
imp[order].plot.barh(xerr=sd[order], title='Permutation importance (drop in test AUC)', figsize=(7, 3.5)); plt.tight_layout(); plt.show()

## 7. Conclusions

**Can we predict default? Yes, moderately.** (Figures below are from the run that produced this notebook; re-run to refresh.)

- Logistic regression: CV ROC AUC ~0.777, test ROC AUC ~0.775 (bootstrap 95% CI ~0.74-0.80), PR AUC ~0.34 vs 0.10 for a no-skill model. Brier score improves from 0.094 to 0.082.
- Gradient boosting did **not** beat logistic regression (CV AUC ~0.761, with higher fold variance). The signal is largely additive, so the simpler and more explainable model is the pick.
- Used for ranking, the model is useful: flagging the riskiest 10% of applicants catches ~34% of defaults (~3.4x lift); the riskiest 20% catches ~52%. It is far from a precise classifier - most flagged applicants still would not default - so it suits prioritising review or risk-based pricing rather than automatic accept/reject.
- Main drivers: higher `debt_ratio` and larger `loan_amount_k` raise risk; higher `income_k` and longer `employment_years` lower it. `age`, `credit_lines` and (mostly) `region` add little.

**Caveats**
- Data quality: 643 applicants have `employment_years` greater than `age - 16` (e.g. a 24-year-old with 14 years at the current employer), and 720 have exactly 0 years (default rate ~16% vs ~10.5% overall). Check how these fields are recorded before relying on them.
- Only 837 defaults, so estimates are noisy (see the CI). Single random split; no time ordering is available, so we cannot test stability over time.
- Predictors are only application-time fields; no payment-history data. Results apply to funded loans only (the `defaulted` label does not exist for rejected applicants), so there is possible selection bias.
- Predicted probabilities should be checked for calibration (plot above) before being used as absolute risk estimates.

**Possible next steps:** tune thresholds against actual cost of a missed default vs. a rejected good loan; add interaction/non-linear terms (e.g. splines) for `income_k` and `debt_ratio`; validate on a later cohort.